In [1]:
"""
이름: 왜용
목적: 어린 아이들의 순수한 질문을 학습할 수 있게 변환
핵심 기능: 
  1. 질문을 과학적 해석으로 풀어서 눈높이 설명
  2. 간단한 실험이 가능하면 추천
  3. 학습 기록과 관련 질문 생성

그래프 구조:
[질문 입력]
    ↓
[질문 의도 분석]
    ↓
[학습 주제 변환]
    ↓
[연령/수준 판단]
    ↓
[눈높이 설명 생성]
    ↓
[이해도 확인 질문]
    ↓
[아이 답변 평가]
    ↓
 ┌───────────────┬───────────────┐
 ↓               ↓               ↓
[재설명]      [퀴즈 생성]      [실험 추천]
 ↓               ↓               ↓
[학습 기록 저장 및 관련 질문 추천]
"""

'\n이름: 왜용\n목적: 어린 아이들의 순수한 질문을 학습할 수 있게 변환\n핵심 기능: \n  1. 질문을 과학적 해석으로 풀어서 눈높이 설명\n  2. 간단한 실험이 가능하면 추천\n  3. 학습 기록과 관련 질문 생성\n\n그래프 구조:\n[질문 입력]\n    ↓\n[질문 의도 분석]\n    ↓\n[학습 주제 변환]\n    ↓\n[연령/수준 판단]\n    ↓\n[눈높이 설명 생성]\n    ↓\n[이해도 확인 질문]\n    ↓\n[아이 답변 평가]\n    ↓\n ┌───────────────┬───────────────┐\n ↓               ↓               ↓\n[재설명]      [퀴즈 생성]      [실험 추천]\n ↓               ↓               ↓\n[학습 기록 저장 및 관련 질문 추천]\n'

In [2]:
import json
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict, List
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

llm = init_chat_model(model="openai:gpt-4o-mini")

In [3]:
class QuestionPurpose(BaseModel):
    intent_type: str = Field(description="질문 의도 유형")
    child_curiosity: str = Field(description="아이가 궁금해하는 핵심 내용")
    learning_need: str = Field(description="학습에 필요한 개념")
    emotion: str = Field(description="질문에 담긴 감정")

class StudySubject(BaseModel):
    title: str = Field(description="학습 주제 제목")
    core_concept: str = Field(description="핵심 개념")
    learning_goal: str = Field(description="학습 목표")
    keywords: List[str] = Field(description="관련 키워드 목록")

class Description(BaseModel):
    target_age: str = Field(description="대상 연령")
    description: str = Field(description="눈높이 설명")

class QuestionList(BaseModel):
    question_list: List[str] = Field(description="질문 목록")

class QuestionEvaluation(BaseModel):
    score: int = Field(description="이해도 점수(0~100)")
    understood: List[str] = Field(description="아이가 이해한 부분")
    confused: List[str] = Field(description="헷갈린 부분/오개념 가능성")
    next_step: str = Field(description="다음 단계: re_explain | quiz | experiment")
    confidence: float = Field(description="판단 신뢰도(0~1)")
    feedback: str = Field(description="부모/교사에게 주는 짧은 피드백")

class QuizEvaluation(BaseModel):
    score: int = Field(description="퀴즈 기반 이해도 점수(0~100)")
    understood: List[str] = Field(description="퀴즈에서 잘한 부분")
    confused: List[str] = Field(description="퀴즈에서 헷갈린 부분")
    next_step: str = Field(description="다음 단계: re_explain | experiment")
    confidence: float = Field(description="판단 신뢰도(0~1)")
    feedback: str = Field(description="부모/교사에게 주는 짧은 피드백")

class State(TypedDict, total=False):
    target_age: int
    question: str
    question_purpose: QuestionPurpose
    study_subject: StudySubject
    description: Description
    question_list: QuestionList
    child_answer: str
    question_evaluation_result: QuestionEvaluation
    next_action: str
    re_explanation: str
    quiz: List[str]
    quiz_answer: str
    quiz_evaluation_result: QuizEvaluation
    experiment: List[str]

graph_builder = StateGraph(State)


In [4]:
def ensure_target_age(state: State):
    target_age = state.get("target_age")
    if target_age is None:
        target_age = interrupt({
            "prompt": "아이 나이를 입력해 주세요 (숫자만). 예: 4",
        })

    try:
        target_age_int = int(str(target_age).strip())
    except Exception:
        target_age_int = 4

    return {"target_age": target_age_int}


def question_purpose_analysis(state: State):
    print(f"question_purpose_analysis => {state['question']}")
    response = llm.with_structured_output(QuestionPurpose).invoke(f"""
    당신은 어린이의 순수한 질문을 학습으로 연결하는 교육 에이전트입니다.

    아래 질문의 목적을 분석해주세요.

    질문:
    {state["question"]}
    """)
    return {
        "question_purpose": response
    }


def convert_study_subject(state: State):
    print(f"convert_study_subject => {state['question_purpose']}")
    question_purpose = state["question_purpose"]
    response = llm.with_structured_output(StudySubject).invoke(f"""
    당신은 어린이의 질문을 학습 주제로 변환하는 교육 에이전트입니다.

    원래 질문:
    {state["question"]}

    질문 목적:
    - 의도 유형: {question_purpose.intent_type}
    - 아이의 궁금증: {question_purpose.child_curiosity}
    - 학습 필요 개념: {question_purpose.learning_need}
    - 감정: {question_purpose.emotion}

    위 내용을 바탕으로 어린이가 배울 수 있는 학습 주제를 선정해주세요.
    """)

    return {
        "study_subject": response
    }


def create_description(state: State):
    print(f"create_description => {state['study_subject']}")
    study_subject = state["study_subject"]
    target_age = state.get("target_age", 4)

    response = llm.with_structured_output(Description).invoke(f"""
    당신은 어린이의 질문을 학습 주제로 변환하는 교육 에이전트입니다.

    학습 주제:
    {study_subject.title}

    학습 주제 핵심 개념:
    {study_subject.core_concept}

    학습 목표:
    {study_subject.learning_goal}

    관련 키워드:
    {study_subject.keywords}

    대상 연령: {target_age}세

    위 내용을 바탕으로 대상 연령에 맞는 눈높이 설명을 작성해주세요.

    """)

    return {
        "description": response
    }

def create_question(state: State):
    print(f"create_question => {state['description']}")
    description = state["description"]
    response = llm.with_structured_output(QuestionList).invoke(f"""
    당신은 어린이의 질문을 학습 주제로 변환하는 교육 에이전트입니다.

    눈높이 설명:
    {description.description}

    위 내용을 바탕으로 아이의 이해도를 평가하는 질문을 3가지 만들어주세요.
    """)

    return {
        "question_list": response
    }

def collect_answer(state: State):
    questions = state["question_list"].question_list
    answer = interrupt({
        "questions": questions,
        "instruction": "아래 3개 질문에 아이가 답한 내용을 적어주세요. (번호로 구분해도 좋아요)"
    })

    return {
        "answer": answer
    }

def question_evaluation(state: State):
    print(f"question_evaluation => {state['question_list']}")
    questions = state["question_list"].question_list
    answer = state.get("answer", "")

    response = llm.with_structured_output(QuestionEvaluation).invoke(f"""
    당신은 어린이의 질문을 학습 주제로 변환하는 교육 에이전트입니다.

    이해도 확인 질문:
    {questions}

    아이 답변:
    {answer}

    위 답변을 바탕으로 이해도를 평가해 주세요.

    출력 규칙:
    - score는 0~100 정수
    - next_step은 re_explain | quiz | experiment 중 하나
    - confidence는 0~1 실수
    - understood/confused는 짧은 불릿 문장들로
    """)

    return {
        "question_evaluation_result": response
    }


def decide_next_action(state: State):
    evaluation = state["question_evaluation_result"]

    suggested = (evaluation.next_step or "").strip()
    if evaluation.confidence < 0.6 or suggested not in {"re_explain", "quiz", "experiment"}:
        if evaluation.score < 60:
            action = "re_explain"
        elif evaluation.score < 80:
            action = "quiz"
        else:
            action = "experiment"
    else:
        action = suggested

    return {"next_action": action}


def re_explain(state: State):
    description = state["description"].description
    evaluation = state["question_evaluation_result"]
    target_age = state.get("target_age", 4)

    response = llm.invoke(f"""
    당신은 어린이를 가르치는 선생님입니다.

    기존 설명:
    {description}

    아이가 헷갈린 부분(오개념 가능성):
    {evaluation.confused}

    목표:
    - {target_age}세 눈높이로 더 쉬운 말로 다시 설명
    - 2~3문장 + 아주 간단한 비유 1개
    """)

    return {"re_explanation": response.content}


def create_quiz(state: State):
    study_subject = state["study_subject"]
    evaluation = state["question_evaluation_result"]
    target_age = state.get("target_age", 4)

    response = llm.with_structured_output(QuestionList).invoke(f"""
    당신은 어린이를 위한 퀴즈를 만드는 선생님입니다.

    학습 주제: {study_subject.title}
    핵심 개념: {study_subject.core_concept}

    아이가 헷갈린 부분(오개념 가능성):
    {evaluation.confused}

    {target_age}세 아이가 풀 수 있는 OX/선택형 퀴즈 3개를 만들어 주세요.
    """)

    return {"quiz": response.question_list}


def collect_quiz_answer(state: State):
    quiz = state.get("quiz", [])
    quiz_answer = interrupt({
        "quiz": quiz,
        "instruction": "위 퀴즈 3개에 대한 아이의 답을 적어주세요. (예: 1) O / 2) 2번 / 3) ... )",
    })

    return {"quiz_answer": quiz_answer}


def quiz_evaluation(state: State):
    quiz = state.get("quiz", [])
    quiz_answer = state.get("quiz_answer", "")
    target_age = state.get("target_age", 4)

    response = llm.with_structured_output(QuizEvaluation).invoke(f"""
    당신은 {target_age}세 아이의 퀴즈 답안을 평가하는 선생님입니다.

    퀴즈:
    {quiz}

    아이 답:
    {quiz_answer}

    위 답을 바탕으로 이해도를 평가해 주세요.

    출력 규칙:
    - score는 0~100 정수
    - next_step은 re_explain | experiment 중 하나
    - confidence는 0~1 실수
    - understood/confused는 짧은 불릿 문장들로
    """)

    return {"quiz_evaluation_result": response}


def recommend_experiment(state: State):
    study_subject = state["study_subject"]

    response = llm.invoke(f"""
    당신은 어린이를 위한 간단한 과학 활동을 추천하는 선생님입니다.

    학습 주제: {study_subject.title}
    핵심 개념: {study_subject.core_concept}

    집에서 안전하게 할 수 있는 간단한 실험(활동) 2가지를 추천해 주세요.
    각 활동은 다음 형식으로:
    - 준비물
    - 방법(3단계 이내)
    - 관찰 포인트(한 줄)
    """)

    lines = [line.strip() for line in response.content.splitlines() if line.strip()]
    return {"experiment": lines}


In [5]:
graph_builder.add_node('ensure_target_age', ensure_target_age)

graph_builder.add_node('question_purpose_analysis', question_purpose_analysis)
graph_builder.add_node('convert_study_subject', convert_study_subject)
graph_builder.add_node('create_description', create_description)
graph_builder.add_node('create_question', create_question)
graph_builder.add_node('collect_answer', collect_answer)
graph_builder.add_node('question_evaluation', question_evaluation)

graph_builder.add_node('decide_next_action', decide_next_action)
graph_builder.add_node('re_explain', re_explain)
graph_builder.add_node('create_quiz', create_quiz)
graph_builder.add_node('collect_quiz_answer', collect_quiz_answer)
graph_builder.add_node('quiz_evaluation', quiz_evaluation)
graph_builder.add_node('recommend_experiment', recommend_experiment)

graph_builder.add_edge(START, 'ensure_target_age')
graph_builder.add_edge('ensure_target_age', 'question_purpose_analysis')

graph_builder.add_edge("question_purpose_analysis", 'convert_study_subject')
graph_builder.add_edge("convert_study_subject", 'create_description')
graph_builder.add_edge("create_description", 'create_question')
graph_builder.add_edge("create_question", 'collect_answer')
graph_builder.add_edge("collect_answer", 'question_evaluation')

graph_builder.add_edge("question_evaluation", 'decide_next_action')

def _route(state: State):
    return state["next_action"]

graph_builder.add_conditional_edges(
    "decide_next_action",
    _route,
    {
        "re_explain": "re_explain",
        "quiz": "create_quiz",
        "experiment": "recommend_experiment",
    },
)

graph_builder.add_edge("re_explain", END)

# quiz는 "생성 -> 아이가 풀기 -> 평가"까지 이어집니다.
graph_builder.add_edge("create_quiz", "collect_quiz_answer")
graph_builder.add_edge("collect_quiz_answer", "quiz_evaluation")
graph_builder.add_edge("quiz_evaluation", END)

graph_builder.add_edge("recommend_experiment", END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [10]:
config = {"configurable": {"thread_id": "demo"}}

# 1) target_age를 첫 입력값으로 사용합니다.
# (여기서 target_age를 넣으면 ensure_target_age에서 interrupt가 발생하지 않습니다.)
result = graph.invoke({"question": "왜 1+1=2 인거야?", "target_age": 4}, config=config)
result

question_purpose_analysis => 왜 1+1=2 인거야?
convert_study_subject => intent_type='사고의 시작' child_curiosity='아이가 수학의 기본 원리를 궁금해하는 단계' learning_need='기본 수학 개념과 더하기의 원리' emotion='호기심과 탐구심'
create_description => title='더하기와 수의 기본 개념' core_concept='더하기 (Addition)와 수의 정의' learning_goal='어린이가 더하기의 원리를 이해하고, 1과 1을 더했을 때 2가 되는 이유를 배운다.' keywords=['수학', '더하기', '기본 원리', '수의 개념', '호기심']
create_question => target_age='4세' description='안녕 친구들! 오늘은 숫자와 더하기에 대해서 재미있게 배워볼 거예요.\n\n먼저 숫자는 친구들이 물건을 세거나 나이를 말할 때 쓰는 거예요. 예를 들어, 사과가 1개 있으면 ‘하나’, 사과가 2개 있으면 ‘둘’이라고 해요. 숫자는 우리가 세상을 이해하는 데 아주 중요한 친구랍니다!\n\n이제 ‘더하기’는 숫자를 합치는 거예요. 쉽게 설명해볼게요!\n\n예를 들어, 사과가 1개 있고, 또 사과가 1개 더 있으면, 우리가 가진 사과는 몇 개가 될까요? 맞아요! 1개와 1개를 합치면 2개가 돼요. 그래서 ‘1 더하기 1은 2’라고 해요. \n\n우리가 더하기를 할 때는 두 숫자를 합치는 것을 말하는데, 이건 아주 쉽고 재미있는 수학 놀이처럼 생각하면 돼요!\n\n이제 친구들도 다양한 물건이나 장난감을 가지고 수 숫자를 세어보고, 더하기 놀이를 해보면 정말 즐거울 거예요! \n\n이제 더하기를 잘 배웠으니, 수학이 더 재미있어질 거예요! 같이 해봐요!'


{'target_age': 4,
 'question': '왜 1+1=2 인거야?',
 'question_purpose': QuestionPurpose(intent_type='사고의 시작', child_curiosity='아이가 수학의 기본 원리를 궁금해하는 단계', learning_need='기본 수학 개념과 더하기의 원리', emotion='호기심과 탐구심'),
 'study_subject': StudySubject(title='더하기와 수의 기본 개념', core_concept='더하기 (Addition)와 수의 정의', learning_goal='어린이가 더하기의 원리를 이해하고, 1과 1을 더했을 때 2가 되는 이유를 배운다.', keywords=['수학', '더하기', '기본 원리', '수의 개념', '호기심']),
 'description': Description(target_age='4세', description='안녕 친구들! 오늘은 숫자와 더하기에 대해서 재미있게 배워볼 거예요.\n\n먼저 숫자는 친구들이 물건을 세거나 나이를 말할 때 쓰는 거예요. 예를 들어, 사과가 1개 있으면 ‘하나’, 사과가 2개 있으면 ‘둘’이라고 해요. 숫자는 우리가 세상을 이해하는 데 아주 중요한 친구랍니다!\n\n이제 ‘더하기’는 숫자를 합치는 거예요. 쉽게 설명해볼게요!\n\n예를 들어, 사과가 1개 있고, 또 사과가 1개 더 있으면, 우리가 가진 사과는 몇 개가 될까요? 맞아요! 1개와 1개를 합치면 2개가 돼요. 그래서 ‘1 더하기 1은 2’라고 해요. \n\n우리가 더하기를 할 때는 두 숫자를 합치는 것을 말하는데, 이건 아주 쉽고 재미있는 수학 놀이처럼 생각하면 돼요!\n\n이제 친구들도 다양한 물건이나 장난감을 가지고 수 숫자를 세어보고, 더하기 놀이를 해보면 정말 즐거울 거예요! \n\n이제 더하기를 잘 배웠으니, 수학이 더 재미있어질 거예요! 같이 해봐요!'),
 'question_list': QuestionList(question_list=['사과

In [11]:
# 2) 위 셀 실행 후, 사용자(아이) 답변을 여기에 넣어서 재개합니다.
# 예: "1) ...\n2) ...\n3) ..." 형태로 붙여넣기

child_answer_text = """1) 5개
2) 4
3) 증가해요"""

result2 = graph.invoke(Command(resume=child_answer_text), config=config)
result2

question_evaluation => question_list=['사과가 3개 있을 때, 친구가 2개 더 가져오면 사과는 총 몇 개가 될까요?', "'2 더하기 2는 몇이에요?' 라는 질문에 답할 수 있나요? 그 이유는 무엇인가요?", '주변에 있는 물건들을 세어보면서 두 그룹의 물건을 만들고, 그 그룹의 수를 더해보세요. 결과는 어떻게 될까요?']


{'target_age': 4,
 'question': '왜 1+1=2 인거야?',
 'question_purpose': QuestionPurpose(intent_type='사고의 시작', child_curiosity='아이가 수학의 기본 원리를 궁금해하는 단계', learning_need='기본 수학 개념과 더하기의 원리', emotion='호기심과 탐구심'),
 'study_subject': StudySubject(title='더하기와 수의 기본 개념', core_concept='더하기 (Addition)와 수의 정의', learning_goal='어린이가 더하기의 원리를 이해하고, 1과 1을 더했을 때 2가 되는 이유를 배운다.', keywords=['수학', '더하기', '기본 원리', '수의 개념', '호기심']),
 'description': Description(target_age='4세', description='안녕 친구들! 오늘은 숫자와 더하기에 대해서 재미있게 배워볼 거예요.\n\n먼저 숫자는 친구들이 물건을 세거나 나이를 말할 때 쓰는 거예요. 예를 들어, 사과가 1개 있으면 ‘하나’, 사과가 2개 있으면 ‘둘’이라고 해요. 숫자는 우리가 세상을 이해하는 데 아주 중요한 친구랍니다!\n\n이제 ‘더하기’는 숫자를 합치는 거예요. 쉽게 설명해볼게요!\n\n예를 들어, 사과가 1개 있고, 또 사과가 1개 더 있으면, 우리가 가진 사과는 몇 개가 될까요? 맞아요! 1개와 1개를 합치면 2개가 돼요. 그래서 ‘1 더하기 1은 2’라고 해요. \n\n우리가 더하기를 할 때는 두 숫자를 합치는 것을 말하는데, 이건 아주 쉽고 재미있는 수학 놀이처럼 생각하면 돼요!\n\n이제 친구들도 다양한 물건이나 장난감을 가지고 수 숫자를 세어보고, 더하기 놀이를 해보면 정말 즐거울 거예요! \n\n이제 더하기를 잘 배웠으니, 수학이 더 재미있어질 거예요! 같이 해봐요!'),
 'question_list': QuestionList(question_list=['사과